# Testing the AcademicCloud/GWDG SAIA Chat AI API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/03_llm_basics/notebooks/academiccloud_saia_api_test.ipynb)

This notebook tests the OpenAI-compatible AcademicCloud/GWDG SAIA Chat AI API for teaching and geospatial AI exercises.

API base URL:

```text
https://chat-ai.academiccloud.de/v1
```

Official documentation:

- Institutional access to AI services: https://docs.hpc.gwdg.de/start_here/using_ai_services/contracts_ai/index.html
- SAIA API usage and examples: https://docs.hpc.gwdg.de/services/saia/index.html
- Available Chat AI models: https://docs.hpc.gwdg.de/services/chat-ai/models/index.html

The notebook covers:

1. Student account and API-key application steps
2. Manual API-key entry for the current notebook session
3. Listing available models with `/models`
4. Chat completion with a 10-sentence answer
5. Optional embeddings test
6. Optional vision-language test with a public street image

Do not write API keys into saved notebook cells before committing or sharing the file.

## 1. Student Account and API-Key Setup

For our school/university teaching use, students normally need AcademicCloud/SAIA access before they can run this notebook.

Suggested steps for students:

1. Check that your institution participates in Academic Cloud and that Chat AI/SAIA access is enabled for your group.
2. Create or activate your Academic Cloud account using your institutional identity or email address.
3. Use your Academic Cloud account or Academic ID to request a SAIA/Chat AI API key through the KISSKI/SAIA booking workflow or dashboard.
4. Use the same email address that belongs to your Academic Cloud account when requesting the API key.
5. Wait for the confirmation email with the API endpoint and available model information.
6. Keep the key private. Do not paste it into GitHub, Moodle, screenshots, shared notebooks, or chat messages.
7. API keys may expire; the example confirmation email says keys are valid for 6 months.

Institutional note: the GWDG documentation states that AI services are integrated into Academic Cloud. If an institution is not yet using Academic Cloud, an Academic Cloud Basis contract is needed first. For web access, an Academic ID is required; for API usage, an API key can be requested with that identity.

## 2. Setup

Run the next cell every time you start a fresh notebook session. It asks for the API key manually and hides the input.

In [ ]:
from __future__ import annotations

import base64
import getpass
import json
import textwrap
from typing import Any

import requests
from IPython.display import Image as IPythonImage
from IPython.display import Markdown, display

API_BASE_URL = "https://chat-ai.academiccloud.de/v1"
API_KEY = getpass.getpass("Paste your AcademicCloud/GWDG SAIA API key for this session: ").strip()

if not API_KEY:
    raise RuntimeError("No API key entered. Run this cell again and paste your SAIA API key.")

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

print(f"API base URL: {API_BASE_URL}")
print("API key loaded for this notebook session.")

## 3. Helper Function

The SAIA endpoint follows the OpenAI-style HTTP API for core endpoints. The helper below keeps requests small and prints useful error information when something fails.

In [ ]:
def api_request(method: str, endpoint: str, **kwargs: Any) -> Any:
    """Call the SAIA API and return parsed JSON."""
    url = f"{API_BASE_URL}{endpoint}"
    response = requests.request(method, url, headers=headers, timeout=60, **kwargs)

    if not response.ok:
        print(f"HTTP {response.status_code} for {method} {endpoint}")
        try:
            print(json.dumps(response.json(), indent=2, ensure_ascii=False))
        except ValueError:
            print(response.text[:1000])
        response.raise_for_status()

    return response.json()


def preview_json(data: Any, max_chars: int = 3000) -> None:
    """Pretty-print JSON without flooding the notebook."""
    text = json.dumps(data, indent=2, ensure_ascii=False)
    if len(text) > max_chars:
        text = text[:max_chars] + "\n..."
    print(text)


## 4. List Available Models

The `/models` endpoint returns the model names currently available to your API key. Use one of those names in the chat, embedding, or vision requests below.

In [ ]:
models_response = api_request("GET", "/models")
preview_json(models_response)

In [ ]:
models = models_response.get("data", [])
model_ids = [model.get("id") for model in models if model.get("id")]

print("Available model IDs:")
for model_id in model_ids:
    print("-", model_id)

DEFAULT_CHAT_MODEL = "meta-llama-3.1-8b-instruct"
chat_model = DEFAULT_CHAT_MODEL if DEFAULT_CHAT_MODEL in model_ids else (model_ids[0] if model_ids else DEFAULT_CHAT_MODEL)

if chat_model != DEFAULT_CHAT_MODEL:
    print(f"\nDefault model {DEFAULT_CHAT_MODEL!r} was not listed. Falling back to {chat_model!r}.")
else:
    print(f"\nUsing chat model: {chat_model}")


## 5. Chat Completion Test

This asks for exactly 10 sentences, so it is easier to see whether the answer is complete.

In [ ]:
payload = {
    "model": chat_model,
    "messages": [
        {
            "role": "system",
            "content": "You are a concise teaching assistant for geospatial data analysis.",
        },
        {
            "role": "user",
            "content": (
                "Explain spatial autocorrelation in exactly 10 clear sentences. "
                "Use a geospatial data analysis example."
            ),
        },
    ],
    "temperature": 0.2,
    "max_tokens": 900,
}

chat_response = api_request("POST", "/chat/completions", json=payload)
preview_json(chat_response)

In [ ]:
message = chat_response["choices"][0]["message"]["content"]
display(Markdown(message))

## 6. Optional: Embeddings Test

The confirmation email listed `E5 Mistral 7B Instruct`; the SAIA examples use the API model name `e5-mistral-7b-instruct`. If `/models` shows a different embedding model, update `embedding_model` before running the request.

In [ ]:
embedding_model = "e5-mistral-7b-instruct"
if model_ids and embedding_model not in model_ids:
    print(f"Warning: {embedding_model!r} was not listed by /models. Set embedding_model manually if needed.")

embedding_payload = {
    "model": embedding_model,
    "input": [
        "Mainz has dense cycling infrastructure near the university.",
        "A spatial join can attach census attributes to grid cells.",
    ],
    "encoding_format": "float",
}

embedding_response = api_request("POST", "/embeddings", json=embedding_payload)

vectors = [item["embedding"] for item in embedding_response.get("data", [])]
print(f"Received {len(vectors)} embedding vectors from {embedding_model!r}.")
if vectors:
    print(f"Embedding dimension: {len(vectors[0])}")
    print("First 8 values of the first vector:", vectors[0][:8])

## 7. Optional: Vision-Language Street Image Test

This test downloads a street image stored in this course repository, converts it to a base64 data URL, sends it to a vision-language model, and asks the model to describe visible street features in exactly 10 sentences.

Default repo image URL:

```text
https://raw.githubusercontent.com/yfeng-hsm/KI_Geodatenanalyse_SS26/main/lectures/03_llm_basics/assets/mapillary_zurich_street.jpg
```

The notebook downloads the repo image in Python first and sends the image bytes as base64, because the API request should include the image content.

In [ ]:
STREET_IMAGE_URL = "https://raw.githubusercontent.com/yfeng-hsm/KI_Geodatenanalyse_SS26/main/lectures/03_llm_basics/assets/mapillary_zurich_street.jpg"

image_response = requests.get(STREET_IMAGE_URL, timeout=60)
image_response.raise_for_status()

street_image_mime = image_response.headers.get("Content-Type", "image/jpeg").split(";", 1)[0]
street_image_base64 = base64.b64encode(image_response.content).decode("utf-8")

print(f"Downloaded repo street image as {street_image_mime}")
display(IPythonImage(url=STREET_IMAGE_URL, width=700))


In [ ]:
vision_model_candidates = [
    "qwen2.5-vl-72b-instruct",
    "internvl3.5-30b-a3b",
    "internvl2.5-8b-mpo",
]

vision_model = next((model for model in vision_model_candidates if model in model_ids), vision_model_candidates[0])
if model_ids and vision_model not in model_ids:
    print(f"Warning: {vision_model!r} was not listed by /models. Set vision_model manually if needed.")
else:
    print(f"Using vision model: {vision_model}")

vision_payload = {
    "model": vision_model,
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Describe what you see in this street image in exactly 10 sentences. "
                        "Mention transport infrastructure, land use, people or vehicles if visible, "
                        "and any safety-relevant observations."
                    ),
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:{street_image_mime};base64,{street_image_base64}"
                    },
                },
            ],
        }
    ],
    "temperature": 0.2,
    "max_tokens": 1000,
}

vision_response = api_request("POST", "/chat/completions", json=vision_payload)
vision_answer = vision_response["choices"][0]["message"]["content"]
display(Markdown(vision_answer))

## 8. Common Errors

- `401 Unauthorized`: the API key is missing, expired, or copied incorrectly.
- `404 Not Found`: check that the endpoint starts with `https://chat-ai.academiccloud.de/v1`.
- `429 Too Many Requests`: reduce request frequency or parallel calls.
- Model error: run the `/models` cell again and use an available model ID.
- Vision model error: choose a listed vision-language model from `/models`; model names can change over time.